In [27]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

import xgboost as xgb

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)

SCORING = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}
GRID_SEARCH_KWARGS = dict(
    cv=10,
    scoring=SCORING,
    refit='r2',
    n_jobs=-1,
    return_train_score=True
)
LINEAR_PARAM_GRID = {
    'model__fit_intercept': [True, False],
    'model__positive': [False, True]
}
TREE_PARAM_GRID = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
RF_PARAM_GRID = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
MLP_PARAM_GRID = {
    'model__hidden_layer_sizes': [(100,), (100, 50), (50, 25)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
XGB_PARAM_GRID = {
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [50, 100, 200]
}


In [28]:
data = pd.read_csv("../data/data_yolo.csv", index_col=0)
print(data)

      country time_of_day        lat       long       road_type  \
8459       SE         day  55.604209  13.028574            city   
63960      DE         day  50.936889   6.908079  arterial-urban   
71801      IT         day  41.987014  12.496538         highway   
90649      DE         day  49.182153   9.414737         highway   
48806      SE         day  55.576732  13.013051            city   
...       ...         ...        ...        ...             ...   
11242      SE         day  55.598127  12.975875            city   
90673      PL         day  50.253217  19.823732   smaller-rural   
24373      DE         day  50.931399   6.953686            city   
41597      SE         day  55.608149  13.003458            city   
40353      IT       night  41.918667  12.383545            city   

      road_condition            weather  solar_angle_elevation  month  hour  \
8459          normal             cloudy              36.560248      5    14   
63960         normal               ra

In [29]:
data = data.dropna().reset_index(drop=True)
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 7,985 rows and 38 columns


In [30]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["conf"]

data = data.drop(columns=["iou", "lrp"])
data_indices = data.index.to_numpy()

In [31]:
all_columns = data.columns    
all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 7985
Columns: Index(['brightness', 'camera_distance_from_ground', 'camera_offset',
       'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast',
       'country', 'distortion_magnitude', 'field_view_horizontal',
       'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat',
       'lateral_acceleration', 'lateral_velocity', 'long', 'month',
       'noisiness', 'quality', 'rain', 'relative_humidity_2m',
       'road_condition', 'road_type', 'sharpness', 'snowfall',
       'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather',
       'wind_speed_10m'],
      dtype='object')
Number of columns: 33


In [32]:
numeric_columns = ['conf', 'solar_angle_elevation', 'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']

categorical_columns = ["country", "time_of_day", "road_type", "road_condition", "weather"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'month', 'noisiness', 'quality', 'rain', 'relative_humidity_2m', 'road_condition', 'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather', 'wind_speed_10m']


# Assessor Tests

Split data into train 60% and validation 40%

In [33]:
#X_train, X_test, y_train, y_test = train_test_split(data, test_size=0.4, train_size=0.6, stratify=data["iou"])
train_idx, test_idx = train_test_split(data_indices, test_size=0.4, train_size=0.6)

X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print(X_train.columns)

X: 4791 3194
y: 4791 3194
Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour',
       'forward_acceleration', 'lateral_acceleration', 'forward_velocity',
       'lateral_velocity', 'field_view_horizontal',
       'camera_distance_from_ground', 'camera_pitch_angle',
       'distortion_magnitude', 'camera_offset', 'laplacian', 'quality',
       'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity',
       'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'wind_speed_10m',
       'weather_code', 'conf'],
      dtype='object')


In [34]:
model_results = []

### IoU

#### Baseline

Random Prediction. Mean of metric over training set.

In [35]:
mean_iou_train = np.mean(y_train)
iou_pred_test = np.full_like(y_test, mean_iou_train)
mean_baseline_r2 = r2_score(y_test, iou_pred_test)
mean_baseline_mae = np.mean(np.abs(y_test - iou_pred_test))
mean_baseline_mse = np.mean((y_test - iou_pred_test)**2)
print(f"R2 score of mean baseline: {mean_baseline_r2:.4f}")
print(f"MAE of mean baseline: {mean_baseline_mae:.4f}")
print(f"MSE of mean baseline: {mean_baseline_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Constant Mean Predictor',
    'test_r2': mean_baseline_r2,
    'test_mae': mean_baseline_mae,
    'test_mse': mean_baseline_mse,
})


R2 score of mean baseline: -0.0001
MAE of mean baseline: 0.1131
MSE of mean baseline: 0.0232


#### Linear Reg on Conf

In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_grid = GridSearchCV(
    linear_reg_conf_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_grid.fit(conf_train, y_train)

best_idx = linear_reg_conf_grid.best_index_
mean_train_r2 = linear_reg_conf_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_grid.best_params_}")
best_linear_reg_conf = linear_reg_conf_grid.best_estimator_


Mean CV train r2 score 0.2992
Mean CV test r2 score 0.2953
Mean CV train MAE score 0.0950
Mean CV test MAE score 0.0951
Mean CV train MSE score 0.0157
Mean CV test MSE score 0.0157
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [37]:
y_pred = best_linear_reg_conf.predict(conf_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Univariate Linear Regression (Confidence)',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3416
MAE test score 0.0936
MSE test score 0.0153


#### Linear Regression

Train models with cv and then test.

In [38]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_grid = GridSearchCV(
    linear_reg_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_grid.fit(X_train, y_train)

best_idx = linear_reg_grid.best_index_
mean_train_r2 = linear_reg_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_grid.best_params_}")
best_linear_reg = linear_reg_grid.best_estimator_


Mean CV train r2 score 0.3382
Mean CV test r2 score 0.3113
Mean CV train MAE score 0.0924
Mean CV test MAE score 0.0938
Mean CV train MSE score 0.0148
Mean CV test MSE score 0.0154
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [39]:
y_pred = best_linear_reg.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Linear Regression',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3519
MAE test score 0.0926
MSE test score 0.0150


#### Decision Trees

In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_grid = GridSearchCV(
    decision_tree_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_grid.fit(X_train, y_train)

best_idx = decision_tree_grid.best_index_
mean_train_r2 = decision_tree_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_grid.best_params_}")
best_decision_tree = decision_tree_grid.best_estimator_


Mean CV train r2 score 0.3491
Mean CV test r2 score 0.2971
Mean CV train MAE score 0.0909
Mean CV test MAE score 0.0943
Mean CV train MSE score 0.0146
Mean CV test MSE score 0.0157
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda

In [41]:
y_pred = best_decision_tree.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Decision Tree',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3258
MAE test score 0.0943
MSE test score 0.0156


#### Random Forest

In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_grid = GridSearchCV(
    rand_forest_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_grid.fit(X_train, y_train)

best_idx = rand_forest_grid.best_index_
mean_train_r2 = rand_forest_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_grid.best_params_}")
best_rand_forest = rand_forest_grid.best_estimator_


/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda

Mean CV train r2 score 0.4995
Mean CV test r2 score 0.3333
Mean CV train MAE score 0.0800
Mean CV test MAE score 0.0916
Mean CV train MSE score 0.0112
Mean CV test MSE score 0.0149
Best params: {'model__max_depth': 20, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [43]:
y_pred = best_rand_forest.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Random Forest',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3574
MAE test score 0.0911
MSE test score 0.0149


#### MLP

In [44]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_grid = GridSearchCV(
    mlp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_grid.fit(X_train, y_train)

best_idx = mlp_grid.best_index_
mean_train_r2 = mlp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_grid.best_params_}")
best_mlp = mlp_grid.best_estimator_


Mean CV train r2 score 0.3606
Mean CV test r2 score 0.2728
Mean CV train MAE score 0.0913
Mean CV test MAE score 0.0963
Mean CV train MSE score 0.0143
Mean CV test MSE score 0.0163
Best params: {'model__activation': 'relu', 'model__alpha': 0.001, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.01}


In [45]:
y_pred = best_mlp.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'MLP',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3286
MAE test score 0.0946
MSE test score 0.0156


#### XGBoost

In [46]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_grid.fit(X_train, y_train)


best_idx = xgb_grid.best_index_
mean_train_r2 = xgb_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_grid.best_params_}")
best_xgb = xgb_grid.best_estimator_

Mean CV train r2 score 0.5194
Mean CV test r2 score 0.3148
Mean CV train MAE score 0.0808
Mean CV test MAE score 0.0935
Mean CV train MSE score 0.0108
Mean CV test MSE score 0.0153
Best params: {'model__learning_rate': 0.01, 'model__n_estimators': 200}


In [47]:
y_pred = best_xgb.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'XGBoost',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3303
MAE test score 0.0935
MSE test score 0.0155


### LRP


#### Baseline

Predict Only the Mean


In [48]:
mean_lrp_train = np.mean(y_train_lrp)
lrp_pred_test = np.full_like(y_test_lrp, mean_lrp_train)
mean_lrp_r2 = r2_score(y_test_lrp, lrp_pred_test)
mean_lrp_mae = np.mean(np.abs(y_test_lrp - lrp_pred_test))
mean_lrp_mse = np.mean((y_test_lrp - lrp_pred_test)**2)
print(f"R2 score of random baseline: {mean_lrp_r2:.4f}")
print(f"MAE of random baseline: {mean_lrp_mae:.4f}")
print(f"MSE of random baseline: {mean_lrp_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Constant Mean Predictor',
    'test_r2': mean_lrp_r2,
    'test_mae': mean_lrp_mae,
    'test_mse': mean_lrp_mse,
})


R2 score of random baseline: -0.0001
MAE of random baseline: 0.0964
MSE of random baseline: 0.0175


#### Linear Reg on Conf

In [49]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_lrp_grid = GridSearchCV(
    linear_reg_conf_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_lrp_grid.fit(conf_train, y_train_lrp)

best_idx = linear_reg_conf_lrp_grid.best_index_
mean_train_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_lrp_grid.best_params_}")
best_linear_reg_conf_lrp = linear_reg_conf_lrp_grid.best_estimator_


Mean CV train r2 score 0.3784
Mean CV test r2 score 0.3745
Mean CV train MAE score 0.0760
Mean CV test MAE score 0.0760
Mean CV train MSE score 0.0106
Mean CV test MSE score 0.0106
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [50]:
y_pred = best_linear_reg_conf_lrp.predict(conf_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Univariate Linear Regression (Confidence)',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4008
MAE test score 0.0750
MSE test score 0.0105


#### Linear Regression


Train models with cv and then test.


In [51]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_lrp_grid = GridSearchCV(
    linear_reg_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_lrp_grid.fit(X_train, y_train_lrp)

best_idx = linear_reg_lrp_grid.best_index_
mean_train_r2 = linear_reg_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_lrp_grid.best_params_}")
best_linear_reg_lrp = linear_reg_lrp_grid.best_estimator_


Mean CV train r2 score 0.4199
Mean CV test r2 score 0.3970
Mean CV train MAE score 0.0731
Mean CV test MAE score 0.0741
Mean CV train MSE score 0.0098
Mean CV test MSE score 0.0102
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [52]:
y_pred = best_linear_reg_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Linear Regression',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4205
MAE test score 0.0735
MSE test score 0.0101


#### Decision Trees


In [53]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_lrp_grid = GridSearchCV(
    decision_tree_lrp_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_lrp_grid.fit(X_train, y_train_lrp)

best_idx = decision_tree_lrp_grid.best_index_
mean_train_r2 = decision_tree_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_lrp_grid.best_params_}")
best_decision_tree_lrp = decision_tree_lrp_grid.best_estimator_


Mean CV train r2 score 0.4294
Mean CV test r2 score 0.3738
Mean CV train MAE score 0.0713
Mean CV test MAE score 0.0748
Mean CV train MSE score 0.0097
Mean CV test MSE score 0.0106
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda

In [54]:
y_pred = best_decision_tree_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Decision Tree',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4056
MAE test score 0.0735
MSE test score 0.0104


#### Random Forest


In [55]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_lrp_grid = GridSearchCV(
    rand_forest_lrp_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_lrp_grid.fit(X_train, y_train_lrp)

best_idx = rand_forest_lrp_grid.best_index_
mean_train_r2 = rand_forest_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_lrp_grid.best_params_}")
best_rand_forest_lrp = rand_forest_lrp_grid.best_estimator_


/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/predictable-ad/lib/python3.12/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda

Mean CV train r2 score 0.5318
Mean CV test r2 score 0.4061
Mean CV train MAE score 0.0648
Mean CV test MAE score 0.0725
Mean CV train MSE score 0.0079
Mean CV test MSE score 0.0100
Best params: {'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [56]:
y_pred = best_rand_forest_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Random Forest',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4400
MAE test score 0.0714
MSE test score 0.0098


#### MLP


In [57]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_lrp_grid = GridSearchCV(
    mlp_lrp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_lrp_grid.fit(X_train, y_train_lrp)

best_idx = mlp_lrp_grid.best_index_
mean_train_r2 = mlp_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_lrp_grid.best_params_}")
best_mlp_lrp = mlp_lrp_grid.best_estimator_


Mean CV train r2 score 0.4071
Mean CV test r2 score 0.3376
Mean CV train MAE score 0.0743
Mean CV test MAE score 0.0769
Mean CV train MSE score 0.0101
Mean CV test MSE score 0.0112
Best params: {'model__activation': 'relu', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.01}


In [58]:
y_pred = best_mlp_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'MLP',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4110
MAE test score 0.0732
MSE test score 0.0103


#### XGBoost

In [59]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_lrp_grid = GridSearchCV(
    xgb_lrp_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_lrp_grid.fit(X_train, y_train_lrp)


best_idx = xgb_lrp_grid.best_index_
mean_train_r2 = xgb_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_lrp_grid.best_params_}")
best_xgb_lrp = xgb_lrp_grid.best_estimator_

Mean CV train r2 score 0.6077
Mean CV test r2 score 0.3822
Mean CV train MAE score 0.0624
Mean CV test MAE score 0.0742
Mean CV train MSE score 0.0067
Mean CV test MSE score 0.0105
Best params: {'model__learning_rate': 0.01, 'model__n_estimators': 200}


In [60]:
y_pred = best_xgb_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'XGBoost',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4138
MAE test score 0.0732
MSE test score 0.0102


### Final Model Comparison


In [61]:
results_df = pd.DataFrame(model_results, index=None)
results_df["test_r2"] = results_df["test_r2"].round(4)
results_df["test_mae"] = results_df["test_mae"].round(4)
results_df["test_mse"] = results_df["test_mse"].round(4)
results_df.to_csv("results.csv")
display(results_df)


,target,model,test_r2,test_mae,test_mse
0,IoU,Constant Mean Predictor,-0.0001,0.1131,0.0232
1,IoU,Univariate Linear Regression (Confidence),0.3416,0.0936,0.0153
2,IoU,Linear Regression,0.3519,0.0926,0.0150
3,IoU,Decision Tree,0.3258,0.0943,0.0156
4,IoU,Random Forest,0.3574,0.0911,0.0149
5,IoU,MLP,0.3286,0.0946,0.0156
6,IoU,XGBoost,0.3303,0.0935,0.0155
7,LRP,Constant Mean Predictor,-0.0001,0.0964,0.0175
8,LRP,Univariate Linear Regression (Confidence),0.4008,0.0750,0.0105
9,LRP,Linear Regression,0.4205,0.0735,0.0101
